# Hodge Laplacian in Polar Coordinates

This notebook derives the scalar Hodge Laplacian from the metric and
differential-form operators.  For polar coordinates $(r,\theta)$ with
$r>0$,

$$g=dr^2+r^2d\theta^2.$$

With the codifferential convention used below, the result is the
negative of the usual positive-coordinate Laplace operator.


## Polar metric

Both the covariant and contravariant metrics carry explicit tensor
indices.  This lets Egison contract them automatically in the Hodge
formula.


In [1]:
declare symbol r, θ : MathValue

def N : Integer := 2
def x : Vector MathValue := [| r, θ |]

def g_i_j : Matrix MathValue :=
  [| [| 1, 0 |], [| 0, r ^ 2 |] |]_i_j
def g~i~j : Matrix MathValue :=
  [| [| 1, 0 |], [| 0, r ^ (-2) |] |]~i~j


In [2]:
g_#_#


$\begin{pmatrix} 1 & 0 \\ 0 & r^{2} \\ \end{pmatrix}_{\#\#}^{\;\;}$

## Exterior derivative and Hodge star

The Hodge star combines the Levi-Civita tensor, the inverse metric,
and the volume density $\sqrt{|g|}=r$.


In [3]:
def d (A : Tensor MathValue) : Tensor MathValue :=
  !(flip ∂/∂) x A

def hodge (A : Tensor MathValue) : Tensor MathValue :=
  let k := dfOrder A
   in withSymbols [i, j]
        sqrt (M.det g_#_#) *
        foldl
          (.)
          ((subrefs A (map 1#j_$1 (between 1 k))) .
           (subrefs (ε' N k) (map 1#i_$1 (between 1 N))))
          (map 1#g~(i_$1)~(j_$1) [1..k])


## Codifferential and Laplacian

The codifferential is the metric adjoint of $d$ and can be written in
terms of two Hodge stars.  The degree test selects the appropriate
endpoint formula for zero- and top-degree forms.


In [4]:
def δ (A : Tensor MathValue) : Tensor MathValue :=
  let k := dfOrder A
   in (-1) ^ (N * (k + 1) + 1) * (hodge (d (hodge A)))

def Δ (A : Tensor MathValue) : Tensor MathValue :=
  match (dfOrder A) as integer with
  | #0 -> δ (d A)
  | #N -> d (δ A)
  | _  -> d (δ A) + δ (d A)

def f : MathValue := function (r, θ)


Applying the operator to an arbitrary scalar function leaves its
derivatives symbolic, so the coordinate formula is visible directly.


In [5]:
Δ f


$-\frac{\partial^2 f}{\partial 1^2} - \frac{\partial f}{\partial 1} r^{-1} - \frac{\partial^2 f}{\partial 2^2} r^{-2}$

The displayed expression is

$$
\Delta f=-\left(
  \frac{\partial^2f}{\partial r^2}
  +\frac1r\frac{\partial f}{\partial r}
  +\frac1{r^2}\frac{\partial^2f}{\partial\theta^2}
\right).
$$

The $1/r$ and $1/r^2$ terms are not inserted by hand: they arise from
the determinant and inverse metric inside the Hodge star.
